# 01 · 数据探索：从推文分类到单词级语言模型

旧项目的一条推文对应一个 0/1 标签。本项目只有一本连续的英文小说，没有人工标签；
标签将由文本本身构造：**用前面的单词预测后面的单词**。

本 Notebook 查看原始文件、正文、词频、稀有词和文本长度。
这里只探索数据，不构建训练词表；训练词表将在 02 中仅使用训练集构建。


In [ ]:
import os
import sys
from pathlib import Path

# 支持从项目根目录或 notebooks 目录启动。
# Kaggle 上可在这里显式设置：
# os.environ["TIME_MACHINE_PROJECT_ROOT"] = "/kaggle/working/Time_Machine"
hint = os.environ.get("TIME_MACHINE_PROJECT_ROOT")
if hint:
    candidates = [Path(hint).expanduser().resolve()]
else:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([parent / "Time_Machine", parent])
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "config.py").is_file() and (path / "src/generation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请在 Time_Machine 项目内启动，或设置 TIME_MACHINE_PROJECT_ROOT。")
existing_config = sys.modules.get("config")
if existing_config and Path(existing_config.__file__).resolve() != PROJECT_ROOT / "config.py":
    raise RuntimeError("kernel 中已经导入其他项目的 config，请重启 kernel 再运行本 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from config import ExperimentConfig, get_device, resolve_data_path, resolve_output_dir

data_path = resolve_data_path()
output_dir = resolve_output_dir()
device = get_device("auto")
print("项目根目录:", PROJECT_ROOT)
print("数据文件:", data_path)
print("输出目录:", output_dir)
print("计算设备:", device)


## 1. 原始文件与正文

`pg35.txt` 含 Gutenberg 的前言和许可证，还包含书名、作者和目录。
`read_text()` 会去掉这些附加部分，从第一章正文开始读取。
原始文件不改动，清理在内存中进行。


In [ ]:
from src.preprocessing import read_text, tokenize

raw_text = data_path.read_text(encoding="utf-8-sig")
body = read_text(data_path)
print("原始字符数:", len(raw_text))
print("正文字符数:", len(body))
print("原始文件开头:\n", raw_text[:700])
print("\n正文开头:\n", body[:700])
print("\n正文结尾:\n", body[-350:])


## 2. 一个 token 是一个单词

统一小写，保留词内撇号，忽略数字和标点。`time traveller` 是两个 token；
`don't` 是一个 token；`after-dinner` 会分成 `after` 和 `dinner`。
这种规则便于理解词表与预测，但生成的文本不包含标点。


In [ ]:
from collections import Counter

tokens = tokenize(body)
counts = Counter(tokens)
summary = pd.Series({
    "body_characters": len(body),
    "word_tokens": len(tokens),
    "distinct_words": len(counts),
    "words_occurring_once": sum(count == 1 for count in counts.values()),
    "paragraphs": len([paragraph for paragraph in body.split("\n\n") if paragraph.strip()]),
})
display(summary.to_frame("value"))
print("前 40 个 token:", tokens[:40])


## 3. 常见词与长尾

常见词贡献很多 token，很多不同单词却只出现一次。
单词级模型的输出层宽度等于词表大小；稀有词越多，学习和泛化越困难。
`min_freq` 控制训练词表的收录阈值，不能通过偷看验证集、测试集扩展词表。


In [ ]:
frequency = pd.DataFrame(counts.most_common(), columns=["word", "count"])
display(frequency.head(20))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
top = frequency.head(20)
axes[0].bar(top["word"], top["count"])
axes[0].tick_params(axis="x", rotation=60)
axes[0].set(title="Most common words", ylabel="Count")
axes[1].plot(range(1, len(frequency) + 1), frequency["count"])
axes[1].set(xscale="log", yscale="log", xlabel="Word rank", ylabel="Count", title="Word frequency tail")
fig.tight_layout()
plt.show()


## 4. 句子长度的粗略观察

下面只用于 EDA，以句号、问号、感叹号粗略切句，不是正式分句器。
实际训练会把正文看作连续的单词流，并切成固定长度窗口，而不是一句一个分类样本。


In [ ]:
import re

sentence_lengths = pd.Series([
    len(tokenize(sentence)) for sentence in re.split(r"[.!?]+", body)
    if tokenize(sentence)
])
display(sentence_lengths.describe().to_frame("words_per_sentence"))
plt.figure(figsize=(8, 4))
plt.hist(sentence_lengths, bins=35, edgecolor="black")
plt.title("Approximate sentence lengths")
plt.xlabel("Words")
plt.ylabel("Sentence count")
plt.tight_layout()
plt.show()


## 下一步

在 `02_preprocessing.ipynb` 中按顺序切分正文，只用训练集构建词表，
查看 `<unk>` 的影响；然后在 `03_dataset.ipynb` 中构造 X 和 y。
